# Étape 6 - Incertitudes de modélisation (MultiModel)

Une prévision sans marge d'erreur est peu exploitable. Méthode simple pour obtenir
une incertitude : **perturber à la fois les données et le modèle**, et regarder
la dispersion des prédictions.

## La classe `MultiModel`

Elle entraîne `n_models` copies du modèle. Chaque copie diffère par :

- un **échantillon bootstrap** différent des données (tirage avec remise)
- une **graine aléatoire** différente

C'est un *décorateur* : il enveloppe n'importe quel estimateur scikit-learn.
Il hérite aussi de `PythonModel` pour être compatible **MLflow** (étape suivante du projet).

Conséquences pratiques :

- `predict` prend un argument supplémentaire `context` (imposé par MLflow) → `model.predict(None, X)`
- `predict` renvoie un dataframe avec une colonne par copie (`y_pred_0` … `y_pred_9`) plus `y_pred_simple`
- `plotly_predictions` détecte ce cas et trace une **plage** au lieu d'une courbe

## 1. Importer les librairies

In [1]:
import sys
sys.path.append('..')
import yaml
import logging
import logging.config
import numpy as np
import pandas as pd
pd.set_option('display.min_rows', 500)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 500)
pd.set_option('max_colwidth', 400)

from foodcast.domain.transform import etl
from foodcast.domain.feature_engineering import features_offline, features_online
from foodcast.domain.forecast import span_future, cross_validate, plotly_predictions
from foodcast.domain.multi_model import MultiModel
from sklearn.ensemble import RandomForestRegressor
import foodcast.settings as settings
import plotly.graph_objects as go

with open(settings.LOGGING_CONFIGURATION_FILE, 'r') as f:
    logging.config.dictConfig(yaml.safe_load(f.read()))

%load_ext autoreload
%autoreload 2

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/mlflow/pyfunc/utils/data_validation.py:156: FutureWarning: Model's `predict` method contains invalid parameters: {'X'}. Only the following parameter names are allowed: context, model_input, and params. Note that invalid parameters will no longer be permitted in future versions.
  param_names = _check_func_signature(func, "predict")


************************************************************
USING default value : foodcast.settings.dev
************************************************************


## 2. Reprendre les étapes 1 à 4

In [2]:
# --- Reprise des étapes 1 à 4 ---
# jeu d'entraînement
df = etl(settings.DATA_DIR, 197, 200)
df = features_offline(df)
x_train = df.drop(columns=['cash_in']).set_index('order_date')
y_train = df[['order_date', 'cash_in']].set_index('order_date')['cash_in']

# jeu de prédiction
past = etl(settings.DATA_DIR, 200, 200)
future = span_future(past['order_date'].max())
future = features_online(future, past)
future = future.set_index('order_date')

# modèle simple entraîné sur tout
simple_model = RandomForestRegressor(n_estimators=10, random_state=42)
simple_model.fit(x_train, y_train)
future.head()

2026-09-09 15:32:10 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/infrastructure/extract.py - INFO - extract: shape = (2158, 6)
2026-09-09 15:32:10 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/infrastructure/extract.py - INFO - extract: shape = (3247, 6)
2026-09-09 15:32:10 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - clean: shape = (380, 3)
2026-09-09 15:32:10 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - clean: shape = (565, 3)
2026-09-09 15:32:10 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - merge: shape = (945, 2)
2026-09-09 15:32:10 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - resample: shape = (659, 2)
2026-09-09 15:32:10 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - IN

,day_1,day_2,day_3,day_4,day_5,day_6,hour_cos_1,hour_sin_1,lag_1W
order_date,,,,,,,,,
2018-11-05 00:00:00,False,False,False,False,False,False,1.000000,0.000000,0.0
2018-11-05 01:00:00,False,False,False,False,False,False,0.965926,0.258819,0.0
2018-11-05 02:00:00,False,False,False,False,False,False,0.866025,0.500000,0.0
2018-11-05 03:00:00,False,False,False,False,False,False,0.707107,0.707107,0.0
2018-11-05 04:00:00,False,False,False,False,False,False,0.500000,0.866025,0.0


## 3. Regarder la classe `MultiModel`

In [3]:
MultiModel?

Init signature:
MultiModel(
    estimator: 'Optional[BaseEstimator]' = None,
    n_models: 'int' = 10,
) -> 'None'
Docstring:     
Wrapper of multiple clones of a given estimator. Each clone differs only by:
    - the boostrap sample it is trained on
    - its random state (if any).
Inherits from PythonModel so as to be saved with MLflow.
Inherits BaseEstimator and RegressorMixin so as to fit into
sklearn pipelines and sklearn clone method.

Attributes
----------
estimator : sklearn.BaseEstimator
    Any scikit-learn estimator.
n : int
    Number of perturbed estimators.
estimators : list
    List of fitted estimators.
Init docstring:
Initialize the wrapper model.

Parameters
----------
estimator : BaseEstimator, optional
    Any sklearn model having a random_state attribute, by default None.
n : int, optional
    Number of clones to maintain, by default 10.
File:           ~/Documents/ml_data_base/MlOps_1/foodcast/domain/multi_model.py
Type:           type
Subclasses:     

## 4. Créer un MultiModel de 10 répliques

On enveloppe `simple_model` (le `RandomForestRegressor` créé au-dessus).

In [4]:
multi_model = MultiModel(simple_model, n_models=10)
multi_model

2026-09-09 15:32:16 - foodcast.domain.multi_model - INFO - Instantiate 10 models of type:
RandomForestRegressor(n_estimators=10, random_state=42)


,estimator,RandomForestR...ndom_state=42)
,n_models,10
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None


## 5. Validation croisée temporelle (3 folds)

Même appel que pour le modèle simple : `cross_validate` gère les deux cas.

In [5]:
maes, preds = cross_validate(multi_model, x_train, y_train, n_fold=3)
maes

2026-09-09 15:32:19 - foodcast.domain.multi_model - INFO - Instantiate 10 models of type:
RandomForestRegressor(n_estimators=10, random_state=42)
2026-09-09 15:32:19 - foodcast.domain.multi_model - INFO - fit: X of shape (125, 9) on y - seed: 0
2026-09-09 15:32:19 - foodcast.domain.multi_model - INFO - fit: X of shape (125, 9) on y - seed: 1
2026-09-09 15:32:19 - foodcast.domain.multi_model - INFO - fit: X of shape (125, 9) on y - seed: 2
2026-09-09 15:32:19 - foodcast.domain.multi_model - INFO - fit: X of shape (125, 9) on y - seed: 3
2026-09-09 15:32:19 - foodcast.domain.multi_model - INFO - fit: X of shape (125, 9) on y - seed: 4
2026-09-09 15:32:19 - foodcast.domain.multi_model - INFO - fit: X of shape (125, 9) on y - seed: 5
2026-09-09 15:32:19 - foodcast.domain.multi_model - INFO - fit: X of shape (125, 9) on y - seed: 6
2026-09-09 15:32:19 - foodcast.domain.multi_model - INFO - fit: X of shape (125, 9) on y - seed: 7
2026-09-09 15:32:19 - foodcast.domain.multi_model - INFO - fit

array([[27.08110656, 20.88913934, 24.20844262, 24.10430328, 21.75143443,
        26.5102459 , 23.64901639, 25.89991803, 26.16065574, 26.67983607,
        23.94172131],
       [27.25261202, 27.51784836, 29.20759836, 25.32282787, 21.84553188,
        23.45804645, 20.61391842, 28.40085246, 26.64561475, 24.42422131,
        22.78698907],
       [31.81580055, 33.06295297, 34.37570565, 30.25476776, 33.92853689,
        32.61422473, 32.42855191, 32.53762158, 29.46739754, 32.30692623,
        32.40184836]])

## 6. Moyenne et écart-type des MAEs par fold

`maes` a une ligne par fold et une colonne par réplique. On agrège **par ligne** (`axis=1`).

In [6]:
pd.DataFrame({
    'mae_moyenne': maes.mean(axis=1),
    'mae_ecart_type': maes.std(axis=1),
})

,mae_moyenne,mae_ecart_type
0,24.625075,1.951007
1,25.225096,2.692301
2,32.290394,1.353950


## 7. Tracer les prédictions de validation croisée (plage)

In [7]:
plotly_predictions(preds, y_train)

2026-09-09 15:32:24 - foodcast.domain.forecast - INFO - plotly_predictions: target shape = (491,)
2026-09-09 15:32:24 - foodcast.domain.forecast - INFO - plotly_predictions: predictions shape = (366, 11)


## 8. Regarder le code de `MultiModel.fit`

In [8]:
MultiModel.fit??

Signature: MultiModel.fit(self, X: 'pd.DataFrame', y: 'Optional[pd.Series]' = None) -> 'MultiModel'
Source:   
    def fit(self, X: pd.DataFrame, y: Optional[pd.Series] = None) -> MultiModel:
        """
        Fit all clones and rearrange them into a list.
        The initial estimator is fit apart.

        Parameters
        ----------
        X : pd.DataFrame of shape (n_samples, n_features)
            Training data.
        y : Optional[pd.Series] of shape (n_samples,)
            Training labels, by default None.

        Returns
        -------
        MultiModel
            The model itself.
        """
        X, y = check_X_y(X, np.ravel(y))
        self.single_estimator = clone(self.estimator)
        self.single_estimator.fit(X, y)
        self.estimators = []
        for random_state in range(self.n_models):
            e = clone(self.estimator)
            if hasattr(e, 'random_state'):
                e.set_params(random_state=random_state)
            X_bootstrap, y_b

## 9. Entraîner le MultiModel sur tout le jeu d'entraînement

In [9]:
multi_model.fit(x_train, y_train)

2026-09-09 15:32:31 - foodcast.domain.multi_model - INFO - fit: X of shape (491, 9) on y - seed: 0
2026-09-09 15:32:31 - foodcast.domain.multi_model - INFO - fit: X of shape (491, 9) on y - seed: 1
2026-09-09 15:32:31 - foodcast.domain.multi_model - INFO - fit: X of shape (491, 9) on y - seed: 2
2026-09-09 15:32:31 - foodcast.domain.multi_model - INFO - fit: X of shape (491, 9) on y - seed: 3
2026-09-09 15:32:31 - foodcast.domain.multi_model - INFO - fit: X of shape (491, 9) on y - seed: 4
2026-09-09 15:32:31 - foodcast.domain.multi_model - INFO - fit: X of shape (491, 9) on y - seed: 5
2026-09-09 15:32:31 - foodcast.domain.multi_model - INFO - fit: X of shape (491, 9) on y - seed: 6
2026-09-09 15:32:31 - foodcast.domain.multi_model - INFO - fit: X of shape (491, 9) on y - seed: 7
2026-09-09 15:32:31 - foodcast.domain.multi_model - INFO - fit: X of shape (491, 9) on y - seed: 8
2026-09-09 15:32:31 - foodcast.domain.multi_model - INFO - fit: X of shape (491, 9) on y - seed: 9


,estimator,RandomForestR...ndom_state=42)
,n_models,10
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None


## 10. Regarder le code de `MultiModel.predict`

In [10]:
MultiModel.predict??

Signature: MultiModel.predict(self, context: 'Any', X: 'pd.DataFrame') -> 'pd.DataFrame'
Source:   
    def predict(self, context: Any, X: pd.DataFrame) -> pd.DataFrame:
        """
        Generate predictions for each clone and concatenate the results into a pandas dataframe.
        Predictions are bounded above zero.

        Parameters
        ----------
        context : Any
            Used by MLflow in some cases.
        X : pd.DataFrame of shape (n_samples, n_features)
            Prediction data.

        Returns
        -------
        pd.DataFrame
            Concatenation of each clone predictions.
        """
        check_is_fitted(self, ["single_estimator", "estimators"])
        X_index = X.index
        X = check_array(X)
        preds = np.stack([e.predict(X) for e in self.estimators], axis=1)
        preds = np.maximum(0, preds)
        preds = pd.DataFrame(
            preds,
            index=X_index,
            columns=['y_pred_{}'.format(i) for i in range(self

## 11. Prédire le chiffre d'affaires futur

Attention à l'API : `predict(context, X)`. On passe `None` comme `context`.

In [11]:
y_pred = multi_model.predict(None, future)
y_pred.head(20)

2026-09-09 15:32:36 - foodcast.domain.multi_model - INFO - predict: X of shape (168, 9)


,y_pred_0,y_pred_1,y_pred_2,y_pred_3,y_pred_4,y_pred_5,y_pred_6,y_pred_7,y_pred_8,y_pred_9,y_pred_simple
order_date,,,,,,,,,,,
2018-11-05 00:00:00,0.000,0.000000,0.000,0.00000,0.000000,0.00000,0.000000,0.000,0.000,0.000000,0.000
2018-11-05 01:00:00,0.000,0.000000,0.000,0.00000,0.000000,0.00000,0.000000,0.000,0.000,0.000000,0.000
2018-11-05 02:00:00,0.000,0.000000,0.000,0.00000,0.000000,0.00000,0.000000,0.000,0.000,0.000000,0.000
2018-11-05 03:00:00,0.000,0.000000,0.000,0.00000,0.000000,0.00000,0.000000,0.000,0.000,0.000000,0.000
2018-11-05 04:00:00,0.000,0.000000,0.000,0.00000,0.000000,0.00000,0.000000,0.000,0.000,0.000000,0.000
2018-11-05 05:00:00,0.000,0.000000,0.000,0.00000,0.000000,0.00000,0.000000,0.000,0.000,0.000000,0.000
2018-11-05 06:00:00,0.000,0.000000,0.000,0.00000,0.000000,0.00000,0.000000,0.000,0.000,0.000000,0.000
2018-11-05 07:00:00,0.000,0.000000,0.000,0.00000,0.000000,0.00000,0.000000,0.000,0.000,0.000000,0.000
2018-11-05 08:00:00,0.000,0.000000,0.000,0.00000,0.000000,0.00000,0.000000,0.000,0.000,0.000000,0.000


## 12. Tracer la prévision avec incertitude

`plotly_predictions` trace la plage min/max des 10 répliques : la largeur de la bande
représente l'incertitude du modèle.

In [12]:
plotly_predictions(y_pred)

2026-09-09 15:32:41 - foodcast.domain.forecast - INFO - plotly_predictions: predictions shape = (168, 11)


## Félicitations 🎉

Tu as parcouru tout le pipeline : **ETL → features offline → entraînement + validation
temporelle → features online → prévision → incertitude par bootstrap**.

Le `MultiModel` obtenu est déjà au format MLflow : le notebook suivant du projet
(`notebooks/mlflow_tracking.ipynb`) ajoute le *tracking*, la reproductibilité et le packaging.